# Synthetic Inference Logs for A/B Evaluation

Seeds the inference log table `CFG.fq("evaluation")` (`advisor_risk_evaluation`) — simulated
inference logs joined with actual labels. Two consumers:

- the **Quality Monitor** (`resources/advisor_drift.quality_monitor.yml`) — drift + model quality
- `src/serving/ab_test.py` — scores `@champion` vs `@challenger` to conclude the A/B test

In production, a data engineering process builds this table from the endpoint's inference logs
joined with the actual outcome. This demo seeds it from the training table: each advisor becomes one
logged request, routed 50/50 to pyfunc `@champion` / `@challenger`, scored by the model that served it,
with `at_risk` as the actual label.

- `served_model`, `model_version`, `prediction`, `prediction_probability` record what the endpoint returned.
- `ab_test.py` re-scores **every** row with **both** models, so the comparison uses the same data for each.

**Run twice in the demo:**

1. After the first `deploy.py` (100% champion) — creates the table so `bundle deploy` can create the monitor.
   All rows go to `@champion` (a warning notes there is nothing to A/B compare yet).
2. After the retraining deploy (50/50 with a distinct `@challenger`) — the data `ab_test.py` needs.

⚠️ Demo scaffold only: these rows overlap the training data, so scores are optimistic.
Do not run in the client environment — replace with the real pipeline.

In [0]:
import os
import uuid
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
from src.config import CFG
from src.utils import get_logger

logger = get_logger("notebooks.synthetic_inference_logs")
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(CFG.experiment_name)  # one shared experiment for the whole repo
client = MlflowClient()
LABEL_COL = "at_risk"


def load_inner_sklearn(version: str):
    """Load the sklearn pipeline packaged inside a pyfunc model version (exactly what the endpoint serves)."""
    local_dir = mlflow.artifacts.download_artifacts(f"models:/{CFG.fq_pyfunc_model_name}/{version}")
    return mlflow.sklearn.load_model(os.path.join(local_dir, "artifacts", "sklearn_model"))

In [0]:
# ── Served models (pyfunc aliases) ──
champion_v = client.get_model_version_by_alias(CFG.fq_pyfunc_model_name, CFG.model_champion_alias).version
try:
    challenger_v = client.get_model_version_by_alias(CFG.fq_pyfunc_model_name, CFG.model_challenger_alias).version
except Exception:
    challenger_v = None

served = {"champion": champion_v}
if challenger_v and challenger_v != champion_v:
    served["challenger"] = challenger_v
else:
    logger.warning(
        f"No distinct @{CFG.model_challenger_alias} (challenger={challenger_v}, champion={champion_v}) — "
        "routing 100% to champion. Fine for monitoring; re-run after the retraining deploy for the A/B test."
    )
logger.info(f"Simulating traffic to: {served}")

models = {name: load_inner_sklearn(v) for name, v in served.items()}
feature_cols = list(models["champion"].feature_names_in_)

# ── One logged request per advisor, routed 50/50 like the retraining deploy (or 100% champion) ──
eval_pdf = spark.table(CFG.fq("training")).toPandas()
rng = np.random.default_rng(42)
n = len(eval_pdf)

eval_pdf["served_model"] = rng.choice(list(served.keys()), size=n)
eval_pdf["model_version"] = eval_pdf["served_model"].map(served).astype(str)
eval_pdf["inference_id"] = [str(uuid.uuid5(uuid.NAMESPACE_DNS, f"inf_{i}")) for i in range(n)]
eval_pdf["inference_timestamp"] = pd.Timestamp.now().floor("s") - pd.to_timedelta(
    rng.integers(0, 7 * 24 * 3600, size=n), unit="s"
)

# Prediction from the model that served each request
eval_pdf["prediction"] = 0
eval_pdf["prediction_probability"] = 0.0
for name, model in models.items():
    mask = eval_pdf["served_model"] == name
    if mask.any():
        X = eval_pdf.loc[mask, feature_cols]
        eval_pdf.loc[mask, "prediction"] = model.predict(X).astype(int)
        eval_pdf.loc[mask, "prediction_probability"] = model.predict_proba(X)[:, 1].round(4)

# label + prediction as int32 (Spark INT) — same types as the monitor's baseline (advisor_risk_baseline)
eval_pdf[LABEL_COL] = eval_pdf[LABEL_COL].astype("int32")
eval_pdf["prediction"] = eval_pdf["prediction"].astype("int32")

ordered_cols = (
    ["inference_id", "inference_timestamp", "Salesforce_Id__c", "served_model", "model_version"]
    + feature_cols
    + ["prediction", "prediction_probability", LABEL_COL]
)
(
    spark.createDataFrame(eval_pdf[ordered_cols])
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(CFG.fq("evaluation"))
)

logger.info(
    f"{CFG.fq('evaluation')}: {n} rows | served: {eval_pdf['served_model'].value_counts().to_dict()} "
    f"| actual at_risk rate: {eval_pdf[LABEL_COL].mean():.4f}"
)
display(spark.table(CFG.fq("evaluation")).limit(5))